# Fase 5 — Visualización de resultados NER clínico

Este notebook lee los artefactos de `scripts/04_evaluate_ner.py` y genera las tablas y figuras comparativas para la tesis.

**Diseño experimental:** 3 escenarios (`Baseline 0%`, `Mixto 50%`, `Full 100%`) × 2 modelos (`BETO`, `XLM-R`). El `test.jsonl` es idéntico en los tres escenarios.

**Requisito:** haber ejecutado la Fase 4 para cada combinación. Ejemplo:

```bash
python scripts/04_evaluate_ner.py \
    --model-dir models/beto_augmented_100 \
    --data-dir data/augmented_100 \
    --output-dir results/beto_augmented_100 \
    --bootstrap 1000
```

Las figuras se guardan en `results/` con nombres `05_*.png` y las tablas en `05_*.csv` / `05_*.tex`.


In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RESULTS_DIR = PROJECT_ROOT / "results"

SCENARIO_ORDER = ["Baseline (0%)", "Mixto (50%)", "Full (100%)"]
MODEL_ORDER = ["BETO", "XLM-R"]


def normalize_scenario(summary: dict) -> str:
    raw = f"{summary.get('scenario', '')} {summary.get('data_dir', '')}".lower()
    if "baseline" in raw:
        return "Baseline (0%)"
    if "50" in raw or "mixed" in raw:
        return "Mixto (50%)"
    if "100" in raw or "full" in raw:
        return "Full (100%)"
    return str(summary.get("scenario", "?"))


def normalize_model(summary: dict) -> str:
    raw = f"{summary.get('model_name', '')} {summary.get('model_dir', '')}".lower()
    if "beto" in raw or "bert-base-spanish" in raw:
        return "BETO"
    if "xlm" in raw:
        return "XLM-R"
    return str(summary.get("model_dir", "?"))


def load_runs() -> pd.DataFrame:
    rows = []
    for path in sorted(RESULTS_DIR.glob("*/04_eval_summary.json")):
        s = json.loads(path.read_text(encoding="utf-8"))
        boot = s.get("bootstrap") or {}
        rows.append(
            {
                "run": path.parent.name,
                "model": normalize_model(s),
                "scenario": normalize_scenario(s),
                "micro_precision": s["micro"]["precision"],
                "micro_recall": s["micro"]["recall"],
                "micro_f1": s["micro"]["f1"],
                "macro_f1": s["macro"]["f1"],
                "weighted_f1": s["weighted"]["f1"],
                "n_docs": s.get("n_docs"),
                "n_entities_true": s.get("n_entities_true"),
                "n_entities_pred": s.get("n_entities_pred"),
                "boot_micro_low": (boot.get("micro") or {}).get("ci_low"),
                "boot_micro_high": (boot.get("micro") or {}).get("ci_high"),
                "boot_macro_low": (boot.get("macro") or {}).get("ci_low"),
                "boot_macro_high": (boot.get("macro") or {}).get("ci_high"),
            }
        )
    return pd.DataFrame(rows)


df = load_runs()
if df.empty:
    raise SystemExit(
        f"No hay resultados en {RESULTS_DIR}. Ejecuta primero scripts/04_evaluate_ner.py."
    )

scenario_rank = {s: i for i, s in enumerate(SCENARIO_ORDER)}
model_rank = {m: i for i, m in enumerate(MODEL_ORDER)}
df["_s"] = df["scenario"].map(scenario_rank).fillna(99)
df["_m"] = df["model"].map(model_rank).fillna(99)
df = df.sort_values(["_s", "_m", "run"]).drop(columns=["_s", "_m"]).reset_index(drop=True)
df


In [ ]:
table_micro = df.pivot_table(index="scenario", columns="model", values="micro_f1", aggfunc="mean", sort=False)
table_macro = df.pivot_table(index="scenario", columns="model", values="macro_f1", aggfunc="mean", sort=False)

print("Micro-F1 (seqeval, nivel entidad)")
display(table_micro.round(4))
print("Macro-F1 (seqeval, nivel entidad)")
display(table_macro.round(4))

df.to_csv(RESULTS_DIR / "05_resultados_comparativos.csv", index=False, encoding="utf-8-sig")
print(f"Tabla guardada en {RESULTS_DIR / '05_resultados_comparativos.csv'}")


In [ ]:
def ordered_present(values, preferred):
    present = set(values)
    return [v for v in preferred if v in present] + sorted(present - set(preferred))


def grouped_bars(ax, metric: str, title: str) -> None:
    scenarios = ordered_present(df["scenario"], SCENARIO_ORDER)
    models = ordered_present(df["model"], MODEL_ORDER)
    x = list(range(len(scenarios)))
    width = 0.8 / max(len(models), 1)
    for i, model in enumerate(models):
        vals = [
            df[(df["model"] == model) & (df["scenario"] == s)][metric].mean()
            for s in scenarios
        ]
        offset = (i - (len(models) - 1) / 2) * width
        ax.bar([xi + offset for xi in x], vals, width, label=model)
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric.replace("_", "-").upper())
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.legend()


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
grouped_bars(axes[0], "micro_f1", "Micro-F1 (seqeval)")
grouped_bars(axes[1], "macro_f1", "Macro-F1 (seqeval)")
fig.suptitle("Comparación de F1 por escenario y modelo")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "05_f1_por_escenario.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
KEY_LABELS = ["PAIS", "HOSPITAL", "SEXO_SUJETO_ASISTENCIA", "FECHAS"]
per_entity_rows = []

for path in sorted(RESULTS_DIR.glob("*/04_per_entity.csv")):
    summary_path = path.parent / "04_eval_summary.json"
    if not summary_path.exists():
        continue
    s = json.loads(summary_path.read_text(encoding="utf-8"))
    e = pd.read_csv(path)
    e["run"] = path.parent.name
    e["model"] = normalize_model(s)
    e["scenario"] = normalize_scenario(s)
    per_entity_rows.append(e)

per_entity = pd.concat(per_entity_rows, ignore_index=True) if per_entity_rows else pd.DataFrame()
if per_entity.empty:
    print("No se encontraron 04_per_entity.csv")
else:
    mask = per_entity["label"].isin(KEY_LABELS) | per_entity["label"].str.startswith("ID_")
    key = per_entity[mask].copy()
    key["col"] = key["model"].astype(str) + " | " + key["scenario"].astype(str)
    pivot = key.pivot_table(index="label", columns="col", values="f1", aggfunc="mean", sort=False).fillna(0)
    pivot = pivot.sort_index()

    fig, ax = plt.subplots(
        figsize=(max(8, 0.55 * pivot.shape[1]), 0.45 * pivot.shape[0] + 2)
    )
    im = ax.imshow(pivot.values, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, label="F1")
    ax.set_title("F1 por entidad clave (PAIS, HOSPITAL, SEXO, FECHAS e ID_*)")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "05_f1_entidades_clave.png", dpi=300, bbox_inches="tight")
    plt.show()
    display(pivot.round(3))


In [ ]:
boot = df.dropna(subset=["boot_micro_low", "boot_micro_high"])
if boot.empty:
    print("Sin intervalos bootstrap. Ejecuta la Fase 4 con --bootstrap 1000.")
else:
    labels = boot["model"].astype(str) + " | " + boot["scenario"].astype(str)
    x = list(range(len(boot)))
    y = boot["micro_f1"].to_numpy()
    low = y - boot["boot_micro_low"].to_numpy()
    high = boot["boot_micro_high"].to_numpy() - y

    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(boot)), 4))
    ax.errorbar(x, y, yerr=[low, high], fmt="o", capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Micro-F1")
    ax.set_title("Micro-F1 con IC bootstrap al 95%")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "05_bootstrap_micro_f1.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
latex = table_micro.round(4).to_latex(
    caption="Micro-F1 por escenario y modelo (seqeval, nivel entidad).",
    label="tab:micro_f1",
    escape=False,
)
(RESULTS_DIR / "05_tabla_micro_f1.tex").write_text(latex, encoding="utf-8")
print(latex)


## Guía de lectura para la tesis

1. **Micro-F1** es la métrica principal: pondera cada entidad por igual y resume el rendimiento global.
2. **Macro-F1** muestra si el modelo falla en entidades poco frecuentes; es normal que sea menor que micro-F1.
3. Compara **Baseline vs Mixto vs Full** para el mismo modelo. Si la aumentación ayuda, la mejora debe ser consistente y, idealmente, mayor que el ancho del IC bootstrap.
4. Revisa la tabla por entidad: `PAIS`, `HOSPITAL`, `SEXO_SUJETO_ASISTENCIA` y `FECHAS` son clases clave. En `ID_*`, separa el efecto de la aumentación (IDs sintéticos) del rendimiento real.
5. Reporta siempre el test usado. Si el test es MEDDOCAN original, la conclusión correcta es sobre **robustez en español**, no sobre adaptación a entidades peruanas.
6. Para la defensa, incluye: tabla micro/macro, figura por escenario, heatmap por entidad e IC bootstrap.
